# Fundamentals of Machine Learning — Exercise 6
## Decision Trees, Random Forest and Extra Trees

**Format:** independent work, **90 minutes**.

In this exercise you will
- compute the best split of a decision tree **by hand** (in code) on a tiny dataset and build a small tree yourself,
- compare **single-variable** splits (`x1 <= t`) with **two-variable** splits (`x1 + x2 <= t`),
- compare your tree with `scikit-learn`,
- study over-fitting of a single tree on real data (handwritten digits),
- build a **bagging** ensemble yourself and compare **decision tree vs. random forest vs. extra trees**.


Tasks marked **⭐** are optional. If you run out of time, skip the ⭐ tasks first.

### How to work
1. Read the text, then **predict** the result before you run a cell.
2. Complete the `TODO`s. A function that is not finished yet raises `NotImplementedError` — that is expected, not a bug.
3. Run the **check** cell below the task: ✅ means correct, ❌ gives a hint what to look at.
4. Stuck for more than 5 minutes? Open the 💡 hint. Still stuck? Leave it, continue, and write the problem down in the last section.
5. Write short answers into the ✍️ cells. Run the notebook **top to bottom** (cells depend on each other).

## 0. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
RANDOM_STATE = 42


def check(test, ok="Correct.", hint="Not yet."):
    """Run a self-check. `test` is a function returning True/False (an error counts as a failure)."""
    try:
        passed = bool(test())
    except Exception as e:
        passed = False
        hint = hint + f"  [{type(e).__name__}: {e}]"
    print(("✅ " + ok) if passed else ("❌ " + hint))


def plot_regions(predict, X, y, ax, title="", xlim=None, ylim=None, step=0.05):
    """Colour the plane by the class predicted by `predict` and draw the data points."""
    xlim = xlim or (X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
    ylim = ylim or (X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
    gx, gy = np.meshgrid(np.arange(xlim[0], xlim[1], step), np.arange(ylim[0], ylim[1], step))
    Z = np.asarray(predict(np.c_[gx.ravel(), gy.ravel()])).reshape(gx.shape)
    ax.contourf(gx, gy, Z, levels=[-0.5, 0.5, 1.5], cmap="coolwarm", alpha=0.25)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=45, zorder=3)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.grid(False)

## 1. Theory in a nutshell 

### 1.1 What is a decision tree?
A decision tree classifies a sample by asking a sequence of simple yes/no questions about the features.

```
                  [ x1 <= 4.5 ? ]               <- root: a question about ONE feature
                   /           \
                yes             no
        [ x2 <= 7.0 ? ]     [ x2 <= 2.5 ? ]     <- internal nodes
          /       \           /       \
     class 0   class 1   class 0   class 1      <- leaves: predictions
```

* **Internal node** = a test `x_j <= t` on one feature `j` with a threshold `t`.
* **Leaf** = the prediction: the majority class of the training samples that ended up in this leaf.
* **Training** = choosing the questions `(j, t)` — which feature and which threshold.

### 1.2 How does the tree choose a question? — impurity
A node with class proportions $p_1,\dots,p_K$ has the **Gini impurity**

$$G = 1-\sum_{k} p_k^2 \qquad\text{(0 = pure node; 0.5 = 50/50 node with two classes)}$$

or the **entropy** $H=-\sum_k p_k\log_2 p_k$. A split sends $n_L$ samples to the left and $n_R$ to the right child (n in total). The **weighted impurity** after the split and the **gain** are

$$G_{split}=\frac{n_L}{n}G_L+\frac{n_R}{n}G_R,\qquad \Delta G = G_{parent}-G_{split}.$$

The tree tries **every feature** and **every candidate threshold** (midpoints between consecutive distinct sorted values) and takes the split with the **largest gain**. It is *greedy*: it only looks one step ahead.

### 1.3 Growing the tree and over-fitting
The same procedure is repeated recursively in both children until a stopping rule applies: the node is pure, `max_depth` is reached, `min_samples_leaf` / `min_samples_split` forbids a split, or no split has a positive gain.
A tree without limits memorizes the training data (100 % training accuracy) and is **unstable**: a small change of the data can change the tree a lot (**high variance**).

### 1.4 Splits on one variable vs. two variables
A standard tree uses **axis-aligned** splits `x_j <= t` (one variable). Its decision boundary is a "staircase" of horizontal and vertical segments, so a **diagonal** boundary needs many steps. A split on a **combination of two variables**, e.g. `x1 + x2 <= t` (an *oblique* split), can do the same job in one step. `scikit-learn` does not search oblique splits (the search space is much larger), but you can add combined features such as `x1 + x2` by hand.

### 1.5 From one tree to a forest
* **Bagging:** train many trees, each on a **bootstrap sample** (n samples drawn *with replacement*), and let them **vote**. Averaging reduces variance.
* **Random forest = bagging + random feature subsets:** at each split only a random subset of `max_features` features is considered (default $\sqrt{d}$). This makes the trees less correlated, so averaging helps more.
* **Extra trees (Extremely Randomized Trees):** for every candidate feature a **random threshold** is drawn and the best of these random splits is used; by default no bootstrap. More randomness and a cheaper split search.

| | Decision tree | Random forest | Extra trees |
|---|---|---|---|
| training data of a tree | all samples | bootstrap sample | all samples (default) |
| features at a split | all | random subset | random subset |
| threshold | best one | best one | random one |
| variance | high | lower | lowest |
| speed (per tree) | fastest | slower | faster than RF |

**Demo 1 — impurity measures.** Gini and entropy of a two-class node as a function of the proportion $p$ of class 1 (just run it).

In [ ]:
p = np.linspace(0.001, 0.999, 300)
gini_curve = 1 - p**2 - (1 - p)**2
entropy_curve = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

plt.figure(figsize=(6, 3.6))
plt.plot(p, gini_curve, label="Gini  = 1 - p² - (1-p)²")
plt.plot(p, entropy_curve, label="Entropy (bits)")
plt.xlabel("proportion p of class 1 in the node")
plt.ylabel("impurity")
plt.title("Both are 0 for a pure node and maximal for 50/50")
plt.legend()
plt.show()

**Demo 2 — deeper tree, more complicated boundary.** The same noisy data (`make_moons`), trees of growing depth. Look at the staircase shape and at what happens when the depth is not limited (just run it).

In [ ]:
X_m, y_m = make_moons(n_samples=200, noise=0.3, random_state=0)

fig, axs = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, depth in zip(axs, [1, 2, 4, None]):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_m, y_m)
    plot_regions(tree.predict, X_m, y_m, ax, f"max_depth={depth}: {tree.get_n_leaves()} leaves, train acc={tree.score(X_m, y_m):.2f}", step=0.03)
plt.tight_layout()
plt.show()

❓ What happens to the boundary when the depth grows, and why is 100 % training accuracy not good news?

✍️ **Your answer:** *(double-click this cell and write here)*

## 2. Manual split computation on a small dataset 
We use 12 samples with two features `x1`, `x2` and two classes (0 = blue, 1 = red).

In [ ]:
X_small = np.array([[1, 2], [2, 4], [3, 2], [4, 3], [2, 6], [6, 2],
                    [5, 6], [6, 5], [8, 3], [3, 8], [7, 6], [9, 4]], dtype=float)
y_small = np.array([0] * 6 + [1] * 6)

df_small = pd.DataFrame(X_small, columns=["x1", "x2"])
df_small["class"] = y_small
df_small.T

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(X_small[:, 0], X_small[:, 1], c=y_small, cmap="coolwarm", edgecolor="k", s=90)
for i, (a, b) in enumerate(X_small):
    ax.annotate(str(i), (a, b), textcoords="offset points", xytext=(7, 6), fontsize=9)
ax.set_xticks(range(0, 11))
ax.set_yticks(range(0, 10))
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("Tiny dataset (numbers = sample index)")
plt.show()

❓ **Predict:** can you separate the two classes with one vertical or horizontal line? And with one diagonal line? Write your guess.

✍️ **Your answer:** *(double-click this cell and write here)*

### 2.1 Worked example — one split by hand
**Root node:** 6 samples of class 0 and 6 of class 1, so $G_{parent}=1-0.5^2-0.5^2=0.5$.

**Candidate split `x1 <= 4.5`:**

| node | class 0 | class 1 | n | Gini |
|---|---:|---:|---:|---:|
| parent | 6 | 6 | 12 | 0.500 |
| left (`x1 <= 4.5`) | 5 | 1 | 6 | $1-(5/6)^2-(1/6)^2 = 0.278$ |
| right (`x1 > 4.5`) | 1 | 5 | 6 | 0.278 |

$G_{split}=\frac{6}{12}\cdot0.278+\frac{6}{12}\cdot0.278=0.278$, therefore $\Delta G = 0.5-0.278 = 0.222$.

You will now implement this in code, step by step.

### 2.2 Task: Gini impurity
Implement `gini(y)` for an array of class labels.

<details><summary>💡 Hint</summary>

`np.bincount(y)` counts how many times each class occurs. Divide by `len(y)` to get the proportions $p_k$, then return `1 - np.sum(p ** 2)`.

</details>

In [ ]:
def gini(y):
    # Gini impurity of an array of class labels:  1 - sum_k p_k^2
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    # TODO: class proportions p_k (hint: np.bincount), then return 1 - sum(p_k ** 2)
    raise NotImplementedError("Complete gini()")

In [ ]:
check(lambda: np.isclose(gini([0, 0, 1, 1]), 0.5), "a 50/50 node has Gini 0.5", "A 50/50 node must give 0.5.")
check(lambda: np.isclose(gini([1, 1, 1]), 0.0), "a pure node has Gini 0", "A pure node must give 0.")
check(lambda: np.isclose(gini([0, 0, 0, 0, 1]), 1 - 0.8**2 - 0.2**2), "unbalanced node OK", "Expected 1 - 0.8² - 0.2² = 0.32.")
check(lambda: np.isclose(gini(y_small), 0.5), "the root node of our dataset has Gini 0.5", "The root has 6 + 6 samples, so Gini = 0.5.")

### 2.3 Task: gain of a split
Implement `split_gain(X, y, feature, threshold)`: the Gini gain $\Delta G=G_{parent}-G_{split}$ of the split `X[:, feature] <= threshold`. Compare the result with the worked example (0.222).

<details><summary>💡 Hint</summary>

`n = len(y)`; weighted impurity = `len(y_left)/n * gini(y_left) + len(y_right)/n * gini(y_right)`; the gain is `gini(y)` minus the weighted impurity.

</details>

In [ ]:
def split_gain(X, y, feature, threshold):
    # Gini gain of the split  X[:, feature] <= threshold
    left = X[:, feature] <= threshold
    y_left, y_right = y[left], y[~left]
    n = len(y)
    # TODO: weighted impurity of the two children, then return the gain
    raise NotImplementedError("Complete split_gain()")

In [ ]:
check(lambda: np.isclose(split_gain(X_small, y_small, 0, 4.5), 0.2222, atol=1e-3), "x1 <= 4.5 has gain 0.222 (as in the worked example)", "The worked example gives 0.222 for x1 <= 4.5.")
check(lambda: np.isclose(split_gain(X_small, y_small, 1, 4.5), 0.1286, atol=1e-3), "x2 <= 4.5 has gain 0.129", "Expected 0.129 for x2 <= 4.5.")
check(lambda: np.isclose(split_gain(X_small, y_small, 0, 0.5), 0.0), "a split that leaves one child empty has gain 0", "If all samples go to one side, the gain is 0.")

### 2.4 Task: the best split
Implement `best_split(X, y)`. Loop over all features and all candidate thresholds (midpoints between consecutive distinct values — `candidate_thresholds` is provided) and return `(feature, threshold, gain)` of the split with the **largest gain**.

<details><summary>💡 Hint</summary>

Inside the inner loop: `gain = split_gain(X, y, j, t)`; if `gain > best_gain`, remember `j`, `t` and `gain`.

</details>

In [ ]:
def candidate_thresholds(values):
    # midpoints between consecutive distinct sorted values
    u = np.unique(values)
    return (u[:-1] + u[1:]) / 2


def best_split(X, y):
    best_feature, best_threshold, best_gain = None, None, 0.0
    for j in range(X.shape[1]):
        for t in candidate_thresholds(X[:, j]):
            # TODO: compute the gain of this split and remember it if it is the best so far
            raise NotImplementedError("Complete best_split()")
    return best_feature, best_threshold, best_gain

In [ ]:
f, t, g = best_split(X_small, y_small)
print(f"best split: feature x{f + 1} <= {t}, gain = {g:.4f}")
check(lambda: f == 0 and np.isclose(t, 4.5) and np.isclose(g, 0.2222, atol=1e-3), "the best root split is x1 <= 4.5", "Expected x1 <= 4.5 with gain 0.222.")

Now look at **all** candidate splits and their gains (just run it).

In [ ]:
rows = []
for j, name in enumerate(["x1", "x2"]):
    for t in candidate_thresholds(X_small[:, j]):
        rows.append({"feature": name, "threshold": t, "gain": split_gain(X_small, y_small, j, t)})
gains = pd.DataFrame(rows)
print("number of candidate splits:", len(gains))
print(gains.sort_values("gain", ascending=False).head(5).round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.8))
for name, g_ in gains.groupby("feature"):
    ax.plot(g_["threshold"], g_["gain"], "o-", label=name)
ax.set_xlabel("threshold")
ax.set_ylabel("Gini gain")
ax.set_title("Gain of every candidate split")
ax.legend()
plt.show()

❓ (a) How many candidate splits were evaluated and why exactly this number? (b) Which split wins and how big is its advantage over the second best?

✍️ **Your answer:** *(double-click this cell and write here)*

### 2.5 Task: grow a tree
A tree is a split followed by the same procedure in each child. Complete the **recursive** function `build_tree`: the stopping rules are given, the **left** subtree is given as an example, you write the call for the **right** subtree. The prediction and printing helpers are provided.

<details><summary>💡 Hint</summary>

Copy the line for `left` and change the data to the other side: the right child gets `X[~mask], y[~mask]` (`~mask` = all samples that are NOT on the left).

</details>

In [ ]:
def build_tree(X, y, depth=0, max_depth=2):
    majority = int(np.bincount(y, minlength=2).argmax())
    leaf = {"leaf": True, "class": majority, "n": len(y)}
    if depth >= max_depth or np.isclose(gini(y), 0.0):
        return leaf
    j, t, gain = best_split(X, y)
    if j is None:
        return leaf
    mask = X[:, j] <= t
    left = build_tree(X[mask], y[mask], depth + 1, max_depth)    # left subtree (given)
    # TODO: build the right subtree in the same way (use ~mask)
    right = None
    return {"leaf": False, "feature": j, "threshold": t, "gain": gain, "n": len(y), "left": left, "right": right}

In [ ]:
def predict_one(node, x):
    while not node["leaf"]:
        node = node["left"] if x[node["feature"]] <= node["threshold"] else node["right"]
    return node["class"]


def predict_tree(node, X):
    return np.array([predict_one(node, x) for x in X])


def print_tree(node, names=("x1", "x2"), indent=""):
    if node["leaf"]:
        print(f"{indent}class {node['class']}   (n={node['n']})")
        return
    f, t = names[node["feature"]], node["threshold"]
    print(f"{indent}if {f} <= {t:.2f}:   [gain={node['gain']:.3f}, n={node['n']}]")
    print_tree(node["left"], names, indent + "    ")
    print(f"{indent}else:   # {f} > {t:.2f}")
    print_tree(node["right"], names, indent + "    ")


for d in [1, 2, 3]:
    tree_d = build_tree(X_small, y_small, max_depth=d)
    acc = np.mean(predict_tree(tree_d, X_small) == y_small)
    print(f"max_depth={d}: training accuracy = {acc:.3f}")

my_tree = build_tree(X_small, y_small, max_depth=2)
print()
print_tree(my_tree)

In [ ]:
check(lambda: not my_tree["leaf"] and my_tree["feature"] == 0 and np.isclose(my_tree["threshold"], 4.5), "root split is x1 <= 4.5", "The root should be x1 <= 4.5.")
check(lambda: np.mean(predict_tree(my_tree, X_small) == y_small) == 1.0, "a tree of depth 2 classifies all 12 samples correctly", "Depth 2 should reach training accuracy 1.0.")
check(lambda: np.mean(predict_tree(build_tree(X_small, y_small, max_depth=1), X_small) == y_small) == 10 / 12, "depth 1 makes 2 mistakes (accuracy 10/12)", "A single split leaves 2 samples misclassified.")

❓ How many splits (internal nodes) does the tree need to classify the 12 samples perfectly using only axis-aligned splits? Which two features does it use?

✍️ **Your answer:** *(double-click this cell and write here)*

### 2.6 Task: compare with scikit-learn
Fit `DecisionTreeClassifier(criterion="gini", max_depth=2)` on the same data and look at its tree (text and picture). Then **compute the gain of the root split from sklearn's internal arrays** and compare it with your own gain.

Useful attributes of `sk_tree.tree_`: `feature`, `threshold`, `impurity` (Gini of each node), `n_node_samples`, `children_left`, `children_right`. Node 0 is the root.

<details><summary>💡 Hint</summary>

gain = `impurity[0] - (n_left * impurity[left] + n_right * impurity[right]) / n_root`, where `left = children_left[0]`, `right = children_right[0]` and `n_*` are values of `n_node_samples`.

</details>

In [ ]:
sk_tree = DecisionTreeClassifier(criterion="gini", max_depth=2, random_state=0).fit(X_small, y_small)
print(export_text(sk_tree, feature_names=["x1", "x2"]))

fig, ax = plt.subplots(figsize=(8, 5))
plot_tree(sk_tree, feature_names=["x1", "x2"], class_names=["0", "1"], filled=True, impurity=True, ax=ax)
plt.show()

In [ ]:
t_ = sk_tree.tree_
left_id, right_id = t_.children_left[0], t_.children_right[0]
# TODO: Gini gain of the root split computed from sklearn's arrays
sk_root_gain = None
print("sklearn root gain:", sk_root_gain, "| your best_split gain:", g)

In [ ]:
grid = np.random.default_rng(0).uniform(0, 10, size=(3000, 2))
agreement = np.mean(predict_tree(my_tree, grid) == sk_tree.predict(grid))

check(lambda: np.isclose(sk_root_gain, g), "sklearn computes exactly the same gain as your best_split", "The two gains should be equal.")
check(lambda: sk_tree.tree_.feature[0] == my_tree["feature"] and np.isclose(sk_tree.tree_.threshold[0], my_tree["threshold"]), "same root feature and threshold", "The root question should be identical.")
print(f"Your tree and sklearn's tree give the same prediction for {agreement:.1%} of 3000 random points.")

❓ Are the two trees identical? What do `gini`, `samples` and `value` in each node of the sklearn picture mean?

✍️ **Your answer:** *(double-click this cell and write here)*

### 2.7 ⭐ Task: entropy instead of Gini
The **information gain** with entropy is $IG=H(parent)-\left(\frac{n_L}{n}H_L+\frac{n_R}{n}H_R\right)$ with $H=-\sum_k p_k\log_2 p_k$. Implement `entropy(y)` and compute the information gain of the split `x1 <= 4.5`.

<details><summary>💡 Hint</summary>

Drop the zero proportions first: `p = p[p > 0]` (because $0\log 0$ is defined as 0), then `-np.sum(p * np.log2(p))`.

</details>

In [ ]:
def entropy(y):
    y = np.asarray(y)
    # TODO: entropy in bits (watch out for zero proportions)
    raise NotImplementedError("Complete entropy()")


mask = X_small[:, 0] <= 4.5
# TODO: information gain of the split x1 <= 4.5
info_gain = None
print("information gain:", info_gain)

In [ ]:
check(lambda: np.isclose(entropy([0, 0, 1, 1]), 1.0) and np.isclose(entropy([1, 1, 1]), 0.0), "entropy values OK", "Entropy of a 50/50 node is 1 bit, of a pure node 0.")
check(lambda: np.isclose(info_gain, 0.35, atol=1e-2), "information gain of x1 <= 4.5 is about 0.35", "Expected about 0.35 (H(parent)=1, children about 0.65).")
tree_ent = DecisionTreeClassifier(criterion="entropy", max_depth=2, random_state=0).fit(X_small, y_small)
print("entropy-based sklearn tree, root:", f"x{tree_ent.tree_.feature[0] + 1} <= {tree_ent.tree_.threshold[0]}")

❓ Does the choice of impurity measure change the root split here? In practice (large datasets), how different are trees grown with Gini and entropy?

✍️ **Your answer:** *(double-click this cell and write here)*

## 3. Single-variable vs. two-variable splits 

Back to the tiny dataset. The picture compares the trees you know (axis-aligned) with **one** split on a combination of the two variables.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
lims = dict(xlim=(0, 10.5), ylim=(0, 9.5))
for ax, d in zip(axs[:2], [1, 2]):
    tr = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_small, y_small)
    plot_regions(tr.predict, X_small, y_small, ax, f"axis-aligned tree, depth {d} (train acc={tr.score(X_small, y_small):.2f})", **lims)
plot_regions(lambda G: (G.sum(axis=1) > 9.5).astype(int), X_small, y_small, axs[2], "ONE oblique split:  x1 + x2 <= 9.5  (train acc=1.00)", **lims)
plt.tight_layout()
plt.show()

### 3.1 Task: evaluate an oblique split with your own functions
A split on `x1 + x2` is just a split on a **new column** `z = x1 + x2`. Append it to the data, and let your `best_split` search all three columns (`x1`, `x2`, `z`).

<details><summary>💡 Hint</summary>

`X_aug` is given. Call `best_split(X_aug, y_small)` and unpack the result into `feature, threshold, gain`.

</details>

In [ ]:
X_aug = np.c_[X_small, X_small[:, 0] + X_small[:, 1]]    # columns: x1, x2, x1 + x2
# TODO: run best_split on X_aug and unpack (feature, threshold, gain)
feature, threshold, gain = None, None, None
print(feature, threshold, gain)

In [ ]:
check(lambda: feature == 2 and np.isclose(threshold, 9.5), "the best split is on the new column z = x1 + x2 with threshold 9.5", "Expected feature index 2 (x1 + x2) and threshold 9.5.")
check(lambda: np.isclose(gain, 0.5), "gain 0.5 = the whole parent impurity: both children are pure", "A perfect split has a gain equal to the parent Gini, 0.5.")
side = X_aug[:, feature] <= threshold
print("Gini of the children:", gini(y_small[side]), gini(y_small[~side]))

❓ Compare: the axis-aligned tree needed 3 splits for a perfect fit, the oblique one needs 1. Is the oblique tree therefore always better? Think about what the search costs.

✍️ **Your answer:** *(double-click this cell and write here)*

### 3.2 ⭐ Task: which direction is best?
A split on $z=\cos\theta\cdot x_1+\sin\theta\cdot x_2$ uses the direction at angle $\theta$. For $\theta=0^\circ$ it is the axis-aligned split on `x1`, for $\theta=90^\circ$ on `x2`. For every angle $0^\circ,1^\circ,\dots,179^\circ$ find the best gain (your `best_split` on the 1-column array `z`).

<details><summary>💡 Hint</summary>

`z` is given (rounded to avoid floating-point ties). `best_split` needs a 2-D array: `best_split(z.reshape(-1, 1), y_small)` returns `(feature, threshold, gain)` — keep the gain.

</details>

In [ ]:
angles = np.arange(0, 180)
gain_per_angle = []
for a in angles:
    r = np.deg2rad(a)
    z = np.round(np.cos(r) * X_small[:, 0] + np.sin(r) * X_small[:, 1], 9)
    # TODO: best gain for this direction (best_split on z.reshape(-1, 1))
    g_a = 0.0
    gain_per_angle.append(g_a)
gain_per_angle = np.array(gain_per_angle)

In [ ]:
perfect = angles[np.isclose(gain_per_angle, 0.5)]
print(f"perfect separation for angles {perfect.min()}° ... {perfect.max()}°;  gain at 0° = {gain_per_angle[0]:.3f},  at 90° = {gain_per_angle[90]:.3f}")

plt.figure(figsize=(7, 3.6))
plt.plot(angles, gain_per_angle)
plt.axvspan(perfect.min(), perfect.max(), alpha=0.15, color="green", label="perfect split")
plt.axvline(0, color="gray", ls="--"); plt.axvline(90, color="gray", ls="--")
plt.xlabel("direction angle θ (degrees)  [0° = split on x1, 90° = split on x2]")
plt.ylabel("best Gini gain")
plt.legend()
plt.show()

check(lambda: np.isclose(gain_per_angle.max(), 0.5) and 45 in perfect, "direction 45° (x1 + x2) separates the classes perfectly", "Expected maximum gain 0.5 including the angle 45°.")
check(lambda: np.isclose(gain_per_angle[0], 0.2222, atol=1e-3), "angle 0° equals the best x1 split (gain 0.222)", "At 0° the split is on x1 only: gain 0.222.")

❓ Which angles are the axis-aligned splits? How wide is the range of angles that separate the data perfectly and what does that say about how precisely you must choose the direction?

✍️ **Your answer:** *(double-click this cell and write here)*

### 3.3 Task: a bigger example — engineered features
Now 400 noisy points whose true boundary is the diagonal $x_1+x_2=1$ (5 % of labels flipped). Compare ordinary trees with trees that also get the features `x1 + x2` and `x1 - x2`.

In [ ]:
rng = np.random.default_rng(1)
X_diag = rng.uniform(0, 1, size=(400, 2))
y_diag = (X_diag[:, 0] + X_diag[:, 1] > 1).astype(int)
flip = rng.random(400) < 0.05
y_diag = np.where(flip, 1 - y_diag, y_diag)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_diag, y_diag, test_size=0.4, random_state=0, stratify=y_diag)
print(Xd_tr.shape, Xd_te.shape)

**Provided code — read it, then run it.** `add_combinations(X)` appends the two extra columns `x1 + x2` and `x1 - x2`; the loop fits for `max_depth = 1..8` one tree on the raw features and one on the extended features and records the **test accuracy** and the **number of leaves**.

In [ ]:
def add_combinations(X):
    return np.c_[X, X[:, 0] + X[:, 1], X[:, 0] - X[:, 1]]


depths = range(1, 9)
acc_raw, acc_eng, leaves_raw, leaves_eng = [], [], [], []
for d in depths:
    t_raw = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xd_tr, yd_tr)
    t_eng = DecisionTreeClassifier(max_depth=d, random_state=0).fit(add_combinations(Xd_tr), yd_tr)
    acc_raw.append(t_raw.score(Xd_te, yd_te))
    acc_eng.append(t_eng.score(add_combinations(Xd_te), yd_te))
    leaves_raw.append(t_raw.get_n_leaves())
    leaves_eng.append(t_eng.get_n_leaves())

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.3))
axs[0].plot(list(depths), acc_raw, "o-", label="raw features")
axs[0].plot(list(depths), acc_eng, "s-", label="+ x1+x2, x1-x2")
axs[0].set_xlabel("max_depth"); axs[0].set_ylabel("test accuracy"); axs[0].legend()

t_raw5 = DecisionTreeClassifier(max_depth=5, random_state=0).fit(Xd_tr, yd_tr)
t_eng1 = DecisionTreeClassifier(max_depth=1, random_state=0).fit(add_combinations(Xd_tr), yd_tr)
plot_regions(t_raw5.predict, Xd_tr, yd_tr, axs[1], f"raw features, depth 5 ({t_raw5.get_n_leaves()} leaves)", xlim=(0, 1), ylim=(0, 1), step=0.01)
plot_regions(lambda G: t_eng1.predict(add_combinations(G)), Xd_tr, yd_tr, axs[2], "extended features, depth 1 (2 leaves)", xlim=(0, 1), ylim=(0, 1), step=0.01)
plt.tight_layout()
plt.show()

check(lambda: acc_eng[0] > 0.9 and acc_raw[0] < 0.8, "one split on the combined feature beats a depth-1 tree on raw features by far", "Expected: depth 1 on extended features > 0.9, on raw features < 0.8.")
check(lambda: leaves_eng[0] == 2 and leaves_raw[4] > 10, "a staircase needs many leaves to approximate the diagonal", "Depth 1 has 2 leaves; the raw depth-5 tree should have more than 10.")

❓ (a) How deep must the raw-feature tree be to get close to the accuracy of the depth-1 tree on the extended features? (b) Why does sklearn not search oblique splits by itself, and what are the alternatives?

✍️ **Your answer:** *(double-click this cell and write here)*

## 4. Decision tree on real data — handwritten digits 

`load_digits`: 1797 images of 8×8 pixels (64 features, grey values 0–16) of the digits 0–9 (10 classes). A real, medium-size classification problem.

In [ ]:
digits = load_digits()
X_d, y_d = digits.data, digits.target
print("X:", X_d.shape, " classes:", np.unique(y_d), " samples per class:", np.bincount(y_d))

fig, axs = plt.subplots(2, 10, figsize=(14, 3.2))
for ax, img, lab in zip(axs.ravel(), digits.images[:20], y_d[:20]):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lab))
    ax.axis("off")
plt.show()

X_train, X_test, y_train, y_test = train_test_split(X_d, y_d, test_size=0.3, random_state=RANDOM_STATE, stratify=y_d)
print("train:", X_train.shape, " test:", X_test.shape)

### 4.1 Task: an unlimited tree
Fit `DecisionTreeClassifier(random_state=RANDOM_STATE)` **without any limit** on the training set. Report the training accuracy, the test accuracy, the depth and the number of leaves.

<details><summary>💡 Hint</summary>

`full_tree.fit(X_train, y_train)`; `full_tree.score(X, y)` is the accuracy; `get_depth()` and `get_n_leaves()` describe the tree.

</details>

In [ ]:
full_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
# TODO: fit it, then compute train_acc, test_acc, depth, n_leaves
train_acc, test_acc, depth, n_leaves = None, None, None, None
print(f"train accuracy = {train_acc}, test accuracy = {test_acc}, depth = {depth}, leaves = {n_leaves}")

In [ ]:
check(lambda: train_acc == 1.0 and 0.75 < test_acc < 0.92, "training accuracy 100 %, test accuracy clearly lower: over-fitting", "Expected train accuracy 1.0 and test accuracy around 0.85.")

### 4.2 Task: how deep should the tree be?
The loop is **given** (read it!): for `max_depth = 1, ..., 15` it computes (a) the training accuracy and (b) the **5-fold cross-validated accuracy on the training set** (`cross_val_score`). Your task: choose the depth with the best cross-validated accuracy (`best_depth`). The next cell plots both curves and **only then** evaluates that tree once on the test set.

*(The test set must not be used to pick the depth — otherwise it is no longer an honest estimate. Same principle as in Exercise 5.)*

<details><summary>💡 Hint</summary>

`np.argmax(cv_acc)` gives the position of the largest value in the list; the depth at that position is `depths[...]`.

</details>

In [ ]:
depths = list(range(1, 16))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
train_acc_d, cv_acc = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    cv_acc.append(cross_val_score(tree, X_train, y_train, cv=cv).mean())   # CV on the TRAINING set
    tree.fit(X_train, y_train)
    train_acc_d.append(tree.score(X_train, y_train))

# TODO: the depth with the best cross-validated accuracy
best_depth = None

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(depths, train_acc_d, "o-", label="training accuracy")
plt.plot(depths, cv_acc, "s-", label="5-fold CV accuracy (training set)")
plt.axvline(best_depth, color="gray", ls="--", label=f"best depth = {best_depth}")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.show()

best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE).fit(X_train, y_train)
tree_test_acc = best_tree.score(X_test, y_test)
print(f"best depth = {best_depth}: test accuracy = {tree_test_acc:.3f}")

check(lambda: 7 <= best_depth <= 15, "a reasonable depth was chosen", "The CV curve should peak somewhere between depth 7 and 15.")
check(lambda: 0.78 < tree_test_acc < 0.92, "test accuracy of a single tree is around 0.85", "Expected a test accuracy around 0.85.")

A glimpse of the first questions of the tree (pixel names are `pixel_row_column`):

In [ ]:
print(export_text(best_tree, feature_names=list(digits.feature_names), max_depth=2))

❓ (a) Where do training and CV accuracy separate and what does the gap mean? (b) A single tree reaches only ~85 % on a problem that is easy for other models. Why is a tree handicapped on pixel features?

✍️ **Your answer:** *(double-click this cell and write here)*

## 5. Random forest and extra trees 

### 5.1 Demo — what does averaging do to the decision boundary? (just run it)
Same noisy `make_moons` data as in Demo 2: a single unlimited tree vs. a random forest vs. extra trees (100 trees each).

In [ ]:
models_demo = {
    "Decision tree (no limit)": DecisionTreeClassifier(random_state=0),
    "Random forest (100 trees)": RandomForestClassifier(n_estimators=100, random_state=0),
    "Extra trees (100 trees)": ExtraTreesClassifier(n_estimators=100, random_state=0),
}
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (name, mdl) in zip(axs, models_demo.items()):
    mdl.fit(X_m, y_m)
    plot_regions(mdl.predict, X_m, y_m, ax, f"{name}: train acc={mdl.score(X_m, y_m):.2f}", step=0.03)
plt.tight_layout()
plt.show()

❓ How does the boundary of the forests differ from the single tree, and why?

✍️ **Your answer:** *(double-click this cell and write here)*

### 5.2 Task: bagging from scratch
Build a bagging ensemble yourself on the digits data.
1. `bagging_fit`: for each tree draw a **bootstrap sample** (n indices drawn *with replacement* from 0..n-1) and fit a `DecisionTreeClassifier` on it.
2. `bagging_predict` is **given**: all trees predict and the majority class wins. Read it and make sure you understand what `votes` contains.

Your part is only the **bootstrap sample** in `bagging_fit`.

<details><summary>💡 Hint</summary>

Bootstrap = n draws *with replacement* from the indices 0..n-1: `idx = rng.integers(0, n, size=n)`.

</details>

In [ ]:
def bagging_fit(X, y, n_trees=25, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y)
    trees = []
    for b in range(n_trees):
        # TODO: bootstrap indices (n draws with replacement)
        idx = None
        trees.append(DecisionTreeClassifier(random_state=b).fit(X[idx], y[idx]))
    return trees


def bagging_predict(trees, X, k=None):
    # use the first k trees (all trees if k is None)
    votes = np.array([t.predict(X) for t in trees[:k]])      # shape (n_trees, n_samples)
    return np.apply_along_axis(lambda v: np.bincount(v).argmax(), 0, votes)   # majority vote for every sample (given)

In [ ]:
trees = bagging_fit(X_train, y_train, n_trees=50)
ks = list(range(1, 51))
acc_bag = [accuracy_score(y_test, bagging_predict(trees, X_test, k)) for k in ks]
single_tree_acc = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_train, y_train).score(X_test, y_test)

plt.figure(figsize=(7, 4))
plt.plot(ks, acc_bag, label="bagging of k trees")
plt.axhline(single_tree_acc, color="r", ls="--", label=f"one unlimited tree ({single_tree_acc:.3f})")
plt.xlabel("number of trees k"); plt.ylabel("test accuracy"); plt.legend(); plt.show()

check(lambda: acc_bag[24] > single_tree_acc + 0.03, f"25 bagged trees ({acc_bag[24]:.3f}) are clearly better than one tree ({single_tree_acc:.3f})", "Bagging 25 trees should beat the single tree by at least 3 percentage points.")

idx0 = np.random.default_rng(0).integers(0, len(y_train), size=len(y_train))
print(f"a bootstrap sample contains {len(np.unique(idx0)) / len(y_train):.1%} of the distinct training samples")

❓ (a) Why does the accuracy rise with the number of trees but flatten out? (b) What fraction of the training set is in a bootstrap sample (see the printed number) and what happens to the rest?

✍️ **Your answer:** *(double-click this cell and write here)*

### 5.3 Task: decision tree vs. random forest vs. extra trees
Compare the three models from scikit-learn (100 trees each, `random_state=RANDOM_STATE`) on the same split. For each model measure the 5-fold CV accuracy on the training set (mean ± std), the training accuracy, the test accuracy and the time of the fit.

<details><summary>💡 Hint</summary>

`cv_scores = cross_val_score(model, X_train, y_train, cv=cv)`; then `model.fit(X_train, y_train)`; accuracies via `model.score(...)`.

</details>

In [ ]:
models = {
    "Decision tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "Extra trees": ExtraTreesClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

rows = []
for name, model in models.items():
    t0 = time.time()
    # TODO: cv_scores (5-fold, training set) and fit the model on the training set
    raise NotImplementedError("Complete the loop")
    fit_time = time.time() - t0
    rows.append({"model": name,
                 "CV acc (mean)": cv_scores.mean(), "CV acc (std)": cv_scores.std(),
                 "train acc": model.score(X_train, y_train), "test acc": model.score(X_test, y_test),
                 "time [s]": fit_time})
results = pd.DataFrame(rows).set_index("model").round(3)
results

In [ ]:
check(lambda: results.loc["Random forest", "test acc"] > results.loc["Decision tree", "test acc"] + 0.05
              and results.loc["Extra trees", "test acc"] > results.loc["Decision tree", "test acc"] + 0.05,
      "both ensembles beat the single tree by more than 5 percentage points", "Both forests should be clearly better than the single tree.")
check(lambda: (results["train acc"] == 1.0).all(), "all three models fit the training data perfectly - the difference is in generalisation", "Unlimited trees reach training accuracy 1.0.")

rf_oob = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)
print(f"out-of-bag accuracy of a random forest (no extra validation set needed): {rf_oob.oob_score_:.3f}")

❓ (a) All three models have training accuracy 1.0. Why is the test accuracy so different? (b) Which is the fastest and which the slowest model per tree and why?

✍️ **Your answer:** *(double-click this cell and write here)*

### 5.4 How many trees? (just run it)
Test accuracy of the random forest and the extra trees as a function of `n_estimators`. *(Single runs on 540 test samples are noisy — look at the trend, not at tiny differences.)*

In [ ]:
n_list = [1, 2, 5, 10, 25, 50, 100, 200]
curve = {"Random forest": [], "Extra trees": []}
for n_est in n_list:
    curve["Random forest"].append(RandomForestClassifier(n_estimators=n_est, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train).score(X_test, y_test))
    curve["Extra trees"].append(ExtraTreesClassifier(n_estimators=n_est, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train).score(X_test, y_test))

plt.figure(figsize=(7, 4))
for name, vals in curve.items():
    plt.plot(n_list, vals, "o-", label=name)
plt.xscale("log"); plt.xlabel("n_estimators (log scale)"); plt.ylabel("test accuracy"); plt.legend(); plt.show()

❓ How many trees would you use and why is adding more trees never a cause of over-fitting (it only costs time)?

✍️ **Your answer:** *(double-click this cell and write here)*

### 5.5 ⭐ The role of `max_features` (just run it)
Random forest with `max_features = 1, 2, 4, ..., 64` (64 = all pixels = plain bagging). **Predict first:** which setting do you expect to be the best?

In [ ]:
mf_list = [1, 2, 4, 8, 16, 32, 64]
mf_curve = {"Random forest": [], "Extra trees": []}
for mf in mf_list:
    mf_curve["Random forest"].append(RandomForestClassifier(n_estimators=100, max_features=mf, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train).score(X_test, y_test))
    mf_curve["Extra trees"].append(ExtraTreesClassifier(n_estimators=100, max_features=mf, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train).score(X_test, y_test))

plt.figure(figsize=(7, 4))
for name, vals in mf_curve.items():
    plt.plot(mf_list, vals, "o-", label=name)
plt.axvline(8, color="gray", ls="--", label="sqrt(64) = 8 (default)")
plt.xscale("log", base=2); plt.xlabel("max_features"); plt.ylabel("test accuracy"); plt.legend(); plt.show()

❓ Why is `max_features = 64` (all features = plain bagging) the worst for the random forest? What is the trade-off when `max_features` is very small?

✍️ **Your answer:** *(double-click this cell and write here)*

### 5.6 ⭐ Which pixels matter? (just run it)
Impurity-based feature importances of the three models (the sum of the gains of all splits that use the pixel), drawn as 8×8 images.

In [ ]:
imps = {name: models[name].feature_importances_ for name in models}
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (name, imp) in zip(axs, imps.items()):
    im = ax.imshow(imp.reshape(8, 8), cmap="viridis")
    ax.set_title(name)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

❓ How do the importances of the single tree differ from those of the forests? Name one caveat of impurity-based importances.

✍️ **Your answer:** *(double-click this cell and write here)*

## 6. Wrap-up 

### 6.1 Summary table
Fill in the table from your own results (double-click to edit).

| | Decision tree | Random forest | Extra trees |
|---|---|---|---|
| test accuracy | | | |
| training accuracy | | | |
| main source of randomness | | | |
| interpretability | | | |
| when would I use it? | | | |

### 6.2 Questions for discussion
1. Why does a tree not need feature scaling, while gradient descent in Exercise 5 did?
2. A tree of depth 1 is called a *decision stump*. What are its strengths and weaknesses?
3. The tree is built greedily (best split now). Can you think of data where the best first split is not part of the best tree?
4. What is the difference between `max_depth`, `min_samples_leaf` and `min_samples_split` as regularization?
5. For which problem would you choose a single decision tree instead of a random forest, even if the forest were 3 points more accurate?
6. Why does a random forest with 100 trees not over-fit more than one with 10 trees?

### 6.3 Self-check
- [ ] I can compute the Gini impurity and the gain of a split by hand.
- [ ] I can explain which thresholds a tree tries and why only midpoints.
- [ ] I can explain the difference between a single-variable split and a two-variable split.
- [ ] I can explain why an unlimited tree over-fits and how `max_depth` helps.
- [ ] I can explain bagging, the role of `max_features` in a random forest and what is different in extra trees.

